In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path
import zipfile

with zipfile.ZipFile("Bottle.zip", "r") as archive:
    archive.extractall("data")

# Find the bottle folder, even if it is nested, ignoring __MACOSX metadata.
train_folders = [
    path for path in Path("data").rglob("train")
    if path.is_dir() and (path / "good").is_dir() and "__MACOSX" not in path.parts
]

assert len(train_folders) == 1, f"Expected one bottle training folder, found {len(train_folders)}: {train_folders}"
data_root = train_folders[0].parent

print("Dataset folder:", data_root)
print("Normal training images:", len(list((data_root / "train" / "good").glob("*.png"))))
print("Test categories:", sorted(
    path.name for path in (data_root / "test").iterdir()
    if path.is_dir()
))

In [ ]:
import random

# Keep the split the same each time.
SEED = 42

# Find all normal bottle photos.
normal_paths = sorted(
    (data_root / "train" / "good").glob("*.png")
)

assert len(normal_paths) >= 2, "Need at least two normal images."

# Copy the list and mix up the order.
shuffled_paths = normal_paths.copy()
random.Random(SEED).shuffle(shuffled_paths)

# Save 20% for validation.
validation_count = max(1, round(len(shuffled_paths) * 0.2))

val_paths = shuffled_paths[:validation_count]
train_paths = shuffled_paths[validation_count:]

# Check that no file is in both groups.
assert set(train_paths).isdisjoint(val_paths)

# Check that we included every file.
assert len(train_paths) + len(val_paths) == len(normal_paths)

print("Training images:", len(train_paths))
print("Validation images:", len(val_paths))
print("Split checks passed")

In [ ]:
from PIL import Image  # Loads the tool for opening and resizing photos.
import numpy as np  # Loads NumPy to represent photos as arrays of numbers.
import matplotlib.pyplot as plt  # Loads the tool for displaying photos.
import torch  # Loads PyTorch to work with tensors.

def load_image(path):  # Creates a reusable function that takes a photo's location.
    with Image.open(path) as image:  # Opens the photo and closes the file automatically afterward.
        image = image.convert("RGB")  # Gives the photo three color channels: red, green, and blue.
        image = image.resize((128, 128), Image.Resampling.BILINEAR)  # Resizes the photo to 128 pixels wide and tall.
        pixels = np.array(image, dtype=np.float32) / 255.0  # Converts the photo into decimal numbers between 0 and 1.

    return torch.from_numpy(pixels).permute(2, 0, 1).contiguous()  # Returns a PyTorch tensor arranged as colors, height, width.

image_tensor = load_image(train_paths[0])  # Loads and prepares the first photo in our training list.

assert image_tensor.shape == (3, 128, 128)  # Checks that the photo has the expected dimensions.
assert torch.isfinite(image_tensor).all()  # Checks that no values are missing or infinite.
assert 0 <= image_tensor.min() and image_tensor.max() <= 1  # Checks that every value is between 0 and 1.

print("Image shape:", image_tensor.shape)  # Prints the number of color channels, height, and width.
print("Smallest value:", image_tensor.min().item())  # Prints the lowest color value in the photo.
print("Largest value:", image_tensor.max().item())  # Prints the highest color value in the photo.

plt.imshow(image_tensor.permute(1, 2, 0).numpy())  # Rearranges the tensor into the format needed to display the photo.
plt.title("Normal training bottle")  # Adds a title above the photo.
plt.axis("off")  # Hides the graph axes.
plt.show()  # Displays the photo.

In [ ]:
from torch.utils.data import Dataset, DataLoader  # Loads tools for organizing and batching images.

class BottleDataset(Dataset):  # Defines our collection of bottle photos.
    def __init__(self, paths):  # Runs when we create a dataset.
        self.paths = list(paths)  # Stores the photo locations.

    def __len__(self):  # Tells PyTorch how many photos we have.
        return len(self.paths)  # Returns the photo count.

    def __getitem__(self, index):  # Tells PyTorch how to get one photo.
        return load_image(self.paths[index])  # Opens and prepares that photo.

train_dataset = BottleDataset(train_paths)  # Creates the training collection.
val_dataset = BottleDataset(val_paths)  # Creates the validation collection.

generator = torch.Generator().manual_seed(SEED)  # Makes training shuffles repeatable.

train_loader = DataLoader(
    train_dataset,
    batch_size=16,  # Gives the model up to 16 photos at a time.
    shuffle=True,  # Mixes the training order each time through the dataset.
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,  # Also checks validation photos in groups of 16.
    shuffle=False  # Keeps validation order fixed.
)

batch = next(iter(train_loader))  # Gets one training batch.

assert batch.shape == (16, 3, 128, 128)  # Checks its dimensions.
assert torch.isfinite(batch).all()  # Checks that every number is finite.

print("Batch shape:", batch.shape)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
import torch.nn as nn  # Loads PyTorch's neural network building blocks.

torch.manual_seed(SEED)  # Makes the starting model weights repeatable.

class BottleAutoencoder(nn.Module):  # Defines our neural network.
    def __init__(self):
        super().__init__()  # Sets up PyTorch's model features.

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, images):  # Defines how images move through the model.
        compressed = self.encoder(images)  # Compresses the images.
        reconstructed = self.decoder(compressed)  # Recreates the images.
        return reconstructed  # Returns the recreated images.

model = BottleAutoencoder()  # Creates the model on CPU.

with torch.no_grad():  # Checks the model without calculating training gradients.
    output = model(batch)

assert output.shape == batch.shape  # Checks that output and input sizes match.
assert torch.isfinite(output).all()  # Checks for invalid numbers.
assert output.min() >= 0 and output.max() <= 1  # Checks the output range.

print("Input shape:", batch.shape)
print("Output shape:", output.shape)
print("Model checks passed")

In [ ]:
import copy

# Use a separate model so this check doesn't change our main model.
check_model = copy.deepcopy(model)

loss_function = nn.MSELoss()  # Measures the average squared pixel difference.
optimizer = torch.optim.Adam(check_model.parameters(), lr=0.001)

before = next(check_model.parameters()).detach().clone()  # Copies starting weights.

check_model.train()  # Puts the model in training mode.
optimizer.zero_grad()  # Clears any previous gradients.

prediction = check_model(batch)  # Tries to recreate the photos.
loss = loss_function(prediction, batch)  # Compares recreations with originals.

assert torch.isfinite(loss), "Loss is invalid."

loss.backward()  # Calculates how weights should change to reduce the error.

gradients = [p.grad for p in check_model.parameters() if p.requires_grad]
assert all(g is not None for g in gradients), "A gradient is missing."
assert all(torch.isfinite(g).all() for g in gradients), "A gradient is invalid."

optimizer.step()  # Updates the weights using those gradients.

after = next(check_model.parameters()).detach()
assert not torch.equal(before, after), "Weights did not change."

print("Reconstruction loss:", loss.item())
print("Gradient checks passed")
print("Weights changed: training step works")

In [ ]:
import copy

# Start with repeatable random weights and photo order.
torch.manual_seed(SEED)
generator.manual_seed(SEED)

model = BottleAutoencoder()

loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
best_val_loss = float("inf")
best_weights = None
history = []

for epoch in range(EPOCHS):
    # Learn from the 167 training photos.
    model.train()
    training_total = 0.0

    for images in train_loader:
        optimizer.zero_grad()

        recreated = model(images)
        loss = loss_function(recreated, images)

        assert torch.isfinite(loss), "Training error is invalid."

        loss.backward()
        optimizer.step()

        training_total += loss.item() * images.size(0)

    train_loss = training_total / len(train_dataset)

    # Check the 42 validation photos without learning from them.
    model.eval()
    validation_total = 0.0

    with torch.no_grad():
        for images in val_loader:
            recreated = model(images)
            loss = loss_function(recreated, images)

            assert torch.isfinite(loss), "Validation error is invalid."

            validation_total += loss.item() * images.size(0)

    val_loss = validation_total / len(val_dataset)

    history.append({
        "train_loss": train_loss,
        "val_loss": val_loss
    })

    # Remember the model with the lowest validation error.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())

    print(
        f"Round {epoch + 1}/{EPOCHS} | "
        f"Training error: {train_loss:.5f} | "
        f"Validation error: {val_loss:.5f}",
        flush=True
    )

# Restore and save the best model from this run.
model.load_state_dict(best_weights)
model.eval()

Path("artifacts").mkdir(exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "seed": SEED,
    "image_size": 128,
    "epochs": EPOCHS,
    "best_val_loss": best_val_loss,
    "history": history,
    "train_paths": [str(path) for path in train_paths],
    "val_paths": [str(path) for path in val_paths]
}, "artifacts/bottle_autoencoder.pt")

print("\nTraining finished.")
print("Best validation error:", best_val_loss)
print("Saved to artifacts/bottle_autoencoder.pt")

In [ ]:
model.eval()  # Sets the model to evaluation mode.

images = next(iter(val_loader))[:4]  # Gets four normal validation photos.

with torch.no_grad():  # Makes sure we aren't calculating learning adjustments.
    recreated = model(images)  # Compresses and recreates each photo.

errors = ((recreated - images) ** 2).mean(dim=(1, 2, 3))  # Calculates one error per photo.

fig, axes = plt.subplots(2, 4, figsize=(12, 6))  # Makes space for eight pictures.

for i in range(4):  # Displays each original and its recreation.
    axes[0, i].imshow(images[i].permute(1, 2, 0).numpy())  # Displays the original photo.
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(recreated[i].permute(1, 2, 0).numpy())  # Displays the model's attempt.
    axes[1, i].set_title(f"Recreated\nError: {errors[i].item():.5f}")
    axes[1, i].axis("off")

plt.tight_layout()  # Adjusts spacing.
plt.show()  # Shows the pictures.

In [ ]:
model.eval()  # Checks photos without updating the model.
validation_scores = []  # Stores one reconstruction error per photo.

with torch.no_grad():  # Turns off gradient calculations.
    for images in val_loader:  # Goes through all 42 normal validation photos.
        recreated = model(images)  # Recreates each photo.

        scores = ((recreated - images) ** 2).mean(dim=(1, 2, 3))  # Finds each photo's error.
        validation_scores.extend(scores.tolist())  # Adds those errors to our list.

validation_scores = np.array(validation_scores)  # Converts the list to a NumPy array.

assert len(validation_scores) == 42  # Checks we scored all validation photos.
assert np.isfinite(validation_scores).all()  # Checks the scores are valid.

threshold = float(np.quantile(validation_scores, 0.95))  # Sets our cutoff.

print("Validation photos checked:", len(validation_scores))
print("Defect cutoff:", threshold)
print("Normal photos above cutoff:", int((validation_scores > threshold).sum()))

In [ ]:
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, confusion_matrix
)  # Loads tools for measuring predictions.

test_paths = sorted((data_root / "test").glob("*/*.png"))  # Finds the real test photos.
assert test_paths, "No test photos found."

test_labels = np.array([
    0 if path.parent.name == "good" else 1
    for path in test_paths
])  # Records the known answer: 0 = normal, 1 = defective.

test_dataset = BottleDataset(test_paths)  # Uses the same image preparation as training.
test_loader = DataLoader(
    test_dataset, batch_size=16, shuffle=False
)  # Keeps photos in the same order as their labels.

test_scores = []  # Stores one reconstruction error per test photo.
model.eval()  # Puts the model in evaluation mode.

with torch.no_grad():  # Checks photos without learning from them.
    for images in test_loader:
        recreated = model(images)  # Recreates the test photos.
        scores = ((recreated - images) ** 2).mean(dim=(1, 2, 3))  # Measures each photo's error.
        test_scores.extend(scores.tolist())  # Saves those errors.

test_scores = np.array(test_scores)  # Converts scores into an array.
assert len(test_scores) == len(test_labels)  # Checks every photo has a score.
assert np.isfinite(test_scores).all()  # Checks all scores are valid.
assert set(test_labels.tolist()) == {0, 1}  # Checks both normal and defective photos exist.

test_predictions = (test_scores > threshold).astype(int)  # Flags photos above our fixed cutoff.

print("Normal test photos:", int((test_labels == 0).sum()))
print("Defective test photos:", int((test_labels == 1).sum()))
print("ROC-AUC:", roc_auc_score(test_labels, test_scores))
print("Precision:", precision_score(test_labels, test_predictions, zero_division=0))
print("Recall:", recall_score(test_labels, test_predictions, zero_division=0))
print("F1:", f1_score(test_labels, test_predictions, zero_division=0))

matrix = confusion_matrix(test_labels, test_predictions, labels=[0, 1])
print("\nConfusion matrix:")
print(matrix)
print("Rows = actual [normal, defective]")
print("Columns = predicted [normal, defective]")

In [ ]:
cases = [
    ("Defect caught", 1, 1),  # Actually defective, correctly flagged.
    ("Defect missed", 1, 0),  # Actually defective, incorrectly accepted.
    ("False alarm", 0, 1),  # Actually normal, incorrectly flagged.
    ("Normal accepted", 0, 0)  # Actually normal, correctly accepted.
]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))  # Creates four picture spaces.

for ax, (title, actual, predicted) in zip(axes, cases):
    matches = np.where(
        (test_labels == actual) & (test_predictions == predicted)
    )[0]  # Finds photos with this outcome.

    if len(matches) == 0:  # Handles an outcome with no examples.
        ax.set_title(f"{title}\nNo example")
        ax.axis("off")
        continue

    index = matches[0]  # Picks the first matching photo.
    image = load_image(test_paths[index])  # Loads the original test photo.

    ax.imshow(image.permute(1, 2, 0).numpy())  # Displays the photo.
    ax.set_title(
        f"{title}\n"
        f"{test_paths[index].parent.name}\n"
        f"Error: {test_scores[index]:.5f}"
    )  # Shows its actual category and error score.
    ax.axis("off")

plt.suptitle(f"Fixed defect cutoff: {threshold:.5f}")  # Shows our existing cutoff.
plt.tight_layout()
plt.show()